# GA Portfolio Research Analysis

Comprehensive analysis of autonomous portfolio optimization results:
1. Research progress tracking
2. Covariance matrix visualization
3. Performance comparison: Model Portfolio vs SPY benchmark

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("📊 GA Portfolio Research Analytics")
print("=" * 60)

In [ ]:
# Load research results
try:
    df = pd.read_csv("results.tsv", sep="\t")
    df["composite_score"] = pd.to_numeric(df["composite_score"], errors="coerce")
    df["status"] = df["status"].str.strip().str.upper()
    
    print(f"\n✅ Loaded {len(df)} experiments")
    print(f"Columns: {list(df.columns)}")
    
    # Summary statistics
    counts = df["status"].value_counts()
    print(f"\n📈 Experiment outcomes:")
    for status, count in counts.items():
        print(f"  {status}: {count}")
    
    n_keep = counts.get("KEEP", 0)
    n_discard = counts.get("DISCARD", 0)
    n_decided = n_keep + n_discard
    if n_decided > 0:
        print(f"\n✨ Keep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")
    
    df.head(10)
except FileNotFoundError:
    print("⚠️  results.tsv not found. Run autonomous research first.")
    df = None

In [ ]:
# Research Progress Visualization
if df is not None and len(df) > 0:
    fig, ax = plt.subplots(figsize=(16, 8))
    
    # Filter valid experiments
    valid = df[df["status"] != "CRASH"].copy()
    valid = valid.reset_index(drop=True)
    
    baseline_score = valid.loc[0, "composite_score"]
    
    # Plot discarded experiments
    disc = valid[valid["status"] == "DISCARD"]
    ax.scatter(disc.index, disc["composite_score"],
               c="#e74c3c", s=30, alpha=0.4, zorder=2, label="Discarded", marker='x')
    
    # Plot kept experiments
    kept = valid[valid["status"] == "KEEP"]
    ax.scatter(kept.index, kept["composite_score"],
               c="#2ecc71", s=80, zorder=4, label="Kept", edgecolors="black", linewidths=1)
    
    # Running maximum (best score so far)
    kept_mask = valid["status"] == "KEEP"
    kept_idx = valid.index[kept_mask]
    kept_scores = valid.loc[kept_mask, "composite_score"]
    running_max = kept_scores.cummax()
    ax.step(kept_idx, running_max, where="post", color="#27ae60",
            linewidth=3, alpha=0.8, zorder=3, label="Running best")
    
    # Annotate kept experiments
    for idx, score in zip(kept_idx, kept_scores):
        desc = str(valid.loc[idx, "description"]).strip()
        if len(desc) > 40:
            desc = desc[:37] + "..."
        
        ax.annotate(desc, (idx, score),
                    textcoords="offset points",
                    xytext=(8, 8), fontsize=9,
                    color="#1a7a3a", alpha=0.9,
                    rotation=25, ha="left", va="bottom",
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7, edgecolor="none"))
    
    # Baseline line
    ax.axhline(y=baseline_score, color="#95a5a6", linestyle="--", linewidth=2, 
               alpha=0.6, label=f"Baseline: {baseline_score:.4f}")
    
    n_total = len(df)
    n_kept = len(df[df["status"] == "KEEP"])
    best_score = kept_scores.max()
    improvement = ((best_score - baseline_score) / abs(baseline_score)) * 100
    
    ax.set_xlabel("Experiment #", fontsize=13, fontweight='bold')
    ax.set_ylabel("Composite Score (higher is better)", fontsize=13, fontweight='bold')
    ax.set_title(f"🔬 Research Progress: {n_total} Experiments, {n_kept} Improvements | Best: {best_score:.4f} (+{improvement:.1f}%)", 
                 fontsize=15, fontweight='bold', pad=20)
    ax.legend(loc="lower right", fontsize=11, framealpha=0.9)
    ax.grid(True, alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.savefig("progress.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\n💾 Saved to progress.png")
else:
    print("⚠️  No data to plot")

## Portfolio Analysis

Load universe and fetch historical data for covariance analysis and performance comparison.

In [ ]:
# Load universe and fetch data
from prepare import BENCHMARK_CONFIG, load_universe

universe = BENCHMARK_CONFIG['universe']
val_start = BENCHMARK_CONFIG['val_start']
val_end = BENCHMARK_CONFIG['val_end']

print(f"\n🌍 Universe: {', '.join(universe)}")
print(f"📅 Validation period: {val_start} to {val_end}")

# Fetch price data
print("\n📥 Fetching price data...")
data = yf.download(universe + ['SPY'], start=val_start, end=val_end, progress=False)

if isinstance(data.columns, pd.MultiIndex):
    prices = data['Close']
else:
    prices = data

prices = prices.ffill().dropna()
returns = prices.pct_change().dropna()

print(f"✅ Loaded {len(prices)} days of data")
print(f"   Assets: {len(universe)}")
print(f"   Date range: {prices.index[0].date()} to {prices.index[-1].date()}")

## Covariance Matrix Visualization

Correlation structure of assets in the universe.

In [ ]:
# Compute correlation matrix
corr_matrix = returns[universe].corr()

# Create covariance matrix heatmap
fig, ax = plt.subplots(figsize=(12, 10))

# Create mask for upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

# Plot heatmap
sns.heatmap(corr_matrix, 
            mask=mask,
            annot=True, 
            fmt='.2f',
            cmap='RdYlGn',
            center=0,
            square=True,
            linewidths=1,
            cbar_kws={"shrink": 0.8, "label": "Correlation"},
            vmin=-1, vmax=1,
            ax=ax)

ax.set_title('📊 Asset Correlation Matrix (Validation Period)', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('')
ax.set_ylabel('')

plt.tight_layout()
plt.savefig("covariance_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n💾 Saved to covariance_matrix.png")

# Print correlation statistics
print(f"\n📈 Correlation Statistics:")
print(f"   Mean correlation: {corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)].mean():.3f}")
print(f"   Max correlation: {corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)].max():.3f}")
print(f"   Min correlation: {corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)].min():.3f}")

## Performance Comparison: Model Portfolio vs SPY

Run backtest on best strategy and compare with SPY benchmark.

In [ ]:
# Run backtest on best strategy
from strategy import Chromosome
from prepare import run_backtest

print("\n🚀 Running backtest on best strategy...")

# Create chromosome with best parameters (from latest kept experiment)
chromosome = Chromosome(universe)

# Run backtest
result = run_backtest(chromosome, split='val')

print(f"\n✅ Backtest complete!")
print(f"   Composite Score: {result['composite_score']:.4f}")
print(f"   Sharpe Ratio: {result['sharpe']:.4f}")
print(f"   CAGR: {result['cagr']:.2%}")
print(f"   Max Drawdown: {result['max_drawdown']:.2%}")

In [ ]:
# Calculate SPY benchmark metrics
spy_returns = returns['SPY']
spy_cum_returns = (1 + spy_returns).cumprod()
spy_total_return = spy_cum_returns.iloc[-1] - 1

# SPY metrics
n_years = len(spy_returns) / 252
spy_cagr = (1 + spy_total_return) ** (1 / n_years) - 1
spy_volatility = spy_returns.std() * np.sqrt(252)
spy_sharpe = (spy_returns.mean() * 252) / (spy_volatility + 1e-8)

# Sortino
spy_downside = spy_returns[spy_returns < 0]
spy_downside_std = spy_downside.std() * np.sqrt(252) if len(spy_downside) > 0 else spy_volatility
spy_sortino = (spy_returns.mean() * 252) / (spy_downside_std + 1e-8)

# Max drawdown
spy_cum_max = np.maximum.accumulate(spy_cum_returns)
spy_drawdowns = (spy_cum_returns - spy_cum_max) / spy_cum_max
spy_max_dd = spy_drawdowns.min()

# Calmar ratio
spy_calmar = spy_cagr / abs(spy_max_dd) if spy_max_dd != 0 else 0

# Beta and Alpha (vs itself = 1 and 0)
spy_beta = 1.0
spy_alpha = 0.0

# Final value from $10,000
spy_final_value = 10000 * (1 + spy_total_return)

print(f"\n📊 SPY Benchmark Metrics:")
print(f"   CAGR: {spy_cagr:.2%}")
print(f"   Total Return: {spy_total_return:.2%}")
print(f"   Volatility: {spy_volatility:.2%}")
print(f"   Sharpe: {spy_sharpe:.4f}")
print(f"   Sortino: {spy_sortino:.4f}")
print(f"   Max DD: {spy_max_dd:.2%}")
print(f"   Calmar: {spy_calmar:.4f}")
print(f"   $10,000 → ${spy_final_value:,.0f}")

In [ ]:
# Calculate Model Portfolio metrics with Beta and Alpha
portfolio_cagr = result['cagr']
portfolio_total_return = result['total_return']
portfolio_volatility = result['volatility']
portfolio_sharpe = result['sharpe']
portfolio_sortino = result['sortino']
portfolio_max_dd = result['max_drawdown']

# Calmar ratio
portfolio_calmar = portfolio_cagr / abs(portfolio_max_dd) if portfolio_max_dd != 0 else 0

# Calculate Beta and Alpha (need portfolio returns)
# Reconstruct portfolio returns from backtest
weights = chromosome.decode_weights()
portfolio_returns = (returns[universe] * weights).sum(axis=1)

# Beta: covariance(portfolio, spy) / variance(spy)
covariance = np.cov(portfolio_returns, spy_returns)[0, 1]
spy_variance = spy_returns.var()
portfolio_beta = covariance / spy_variance if spy_variance != 0 else 0

# Alpha: portfolio_return - (risk_free + beta * (spy_return - risk_free))
# Assuming risk_free = 0
portfolio_alpha = (portfolio_returns.mean() * 252) - (portfolio_beta * (spy_returns.mean() * 252))

# Final value from $10,000
portfolio_final_value = 10000 * (1 + portfolio_total_return)

print(f"\n📈 Model Portfolio Metrics:")
print(f"   CAGR: {portfolio_cagr:.2%}")
print(f"   Total Return: {portfolio_total_return:.2%}")
print(f"   Volatility: {portfolio_volatility:.2%}")
print(f"   Beta: {portfolio_beta:.4f}")
print(f"   Alpha: {portfolio_alpha:.2%}")
print(f"   Sharpe: {portfolio_sharpe:.4f}")
print(f"   Sortino: {portfolio_sortino:.4f}")
print(f"   Max DD: {portfolio_max_dd:.2%}")
print(f"   Calmar: {portfolio_calmar:.4f}")
print(f"   $10,000 → ${portfolio_final_value:,.0f}")

In [ ]:
# Create comparison table
comparison_data = {
    'Метрика': [
        'CAGR (%)',
        'Совок. доход (%)',
        'Волатильность год. (%)',
        'Beta',
        'Alpha (%)',
        'Max Drawdown (%)',
        'Sharpe Ratio',
        'Sortino Ratio',
        'Calmar Ratio',
        '$10 000 →'
    ],
    'Model Portfolio': [
        f"{portfolio_cagr * 100:.2f}",
        f"{portfolio_total_return * 100:.2f}",
        f"{portfolio_volatility * 100:.2f}",
        f"{portfolio_beta:.4f}",
        f"{portfolio_alpha * 100:.2f}",
        f"{portfolio_max_dd * 100:.2f}",
        f"{portfolio_sharpe:.4f}",
        f"{portfolio_sortino:.4f}",
        f"{portfolio_calmar:.4f}",
        f"${portfolio_final_value:,.0f}"
    ],
    'SPY': [
        f"{spy_cagr * 100:.2f}",
        f"{spy_total_return * 100:.2f}",
        f"{spy_volatility * 100:.2f}",
        f"{spy_beta:.4f}",
        f"{spy_alpha * 100:.2f}",
        f"{spy_max_dd * 100:.2f}",
        f"{spy_sharpe:.4f}",
        f"{spy_sortino:.4f}",
        f"{spy_calmar:.4f}",
        f"${spy_final_value:,.0f}"
    ]
}

comparison_df = pd.DataFrame(comparison_data)

# Calculate differences
def calc_diff(portfolio_val, spy_val, metric_name):
    try:
        # Remove $ and commas for final value
        if '$' in portfolio_val:
            p = float(portfolio_val.replace('$', '').replace(',', ''))
            s = float(spy_val.replace('$', '').replace(',', ''))
            diff = p - s
            return f"${diff:+,.0f}"
        else:
            p = float(portfolio_val)
            s = float(spy_val)
            diff = p - s
            # For percentages and ratios
            if metric_name in ['CAGR (%)', 'Совок. доход (%)', 'Волатильность год. (%)', 'Alpha (%)', 'Max Drawdown (%)']:
                return f"{diff:+.2f}"
            else:
                return f"{diff:+.4f}"
    except:
        return "-"

comparison_df['Разница'] = [
    calc_diff(comparison_df.loc[i, 'Model Portfolio'], 
              comparison_df.loc[i, 'SPY'],
              comparison_df.loc[i, 'Метрика'])
    for i in range(len(comparison_df))
]

print("\n" + "="*80)
print("📊 СРАВНИТЕЛЬНАЯ ТАБЛИЦА: MODEL PORTFOLIO vs SPY")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

In [ ]:
# Create styled HTML table
from IPython.display import HTML

def color_diff(val):
    """Color positive differences green, negative red"""
    if val == '-':
        return ''
    try:
        # Extract numeric value
        num_val = float(val.replace('$', '').replace(',', '').replace('+', ''))
        if num_val > 0:
            return 'background-color: #d4edda; color: #155724; font-weight: bold'
        elif num_val < 0:
            return 'background-color: #f8d7da; color: #721c24; font-weight: bold'
        else:
            return ''
    except:
        return ''

# Apply styling
styled_df = comparison_df.style\
    .set_properties(**{
        'text-align': 'right',
        'padding': '10px',
        'border': '1px solid #ddd'
    })\
    .set_table_styles([
        {'selector': 'th', 'props': [
            ('background-color', '#2c3e50'),
            ('color', 'white'),
            ('font-weight', 'bold'),
            ('text-align', 'center'),
            ('padding', '12px'),
            ('border', '1px solid #34495e')
        ]},
        {'selector': 'td:first-child', 'props': [
            ('font-weight', 'bold'),
            ('text-align', 'left'),
            ('background-color', '#ecf0f1')
        ]},
        {'selector': '', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('font-family', 'Arial, sans-serif'),
            ('font-size', '14px')
        ]}
    ])\
    .applymap(color_diff, subset=['Разница'])

# Save to HTML
html_output = f"""
<html>
<head>
    <meta charset="UTF-8">
    <title>Portfolio Comparison</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 20px; background-color: #f5f5f5; }}
        h1 {{ color: #2c3e50; text-align: center; }}
        .container {{ max-width: 900px; margin: 0 auto; background: white; padding: 30px; border-radius: 10px; box-shadow: 0 2px 10px rgba(0,0,0,0.1); }}
    </style>
</head>
<body>
    <div class="container">
        <h1>📊 Сравнительный анализ портфеля</h1>
        <p style="text-align: center; color: #7f8c8d; margin-bottom: 30px;">
            Период валидации: {val_start} — {val_end}
        </p>
        {styled_df.to_html()}
    </div>
</body>
</html>
"""

with open('comparison_table.html', 'w', encoding='utf-8') as f:
    f.write(html_output)

print("\n💾 Saved styled table to comparison_table.html")

# Display in notebook
display(HTML(styled_df.to_html()))

In [ ]:
# Summary
print("\n" + "="*80)
print("✅ ANALYSIS COMPLETE")
print("="*80)
print("\n📁 Generated files:")
print("   • progress.png - Research progress visualization")
print("   • covariance_matrix.png - Asset correlation heatmap")
print("   • comparison_table.html - Styled performance comparison table")
print("\n🎯 Key Findings:")
print(f"   • Model Portfolio CAGR: {portfolio_cagr:.2%} vs SPY: {spy_cagr:.2%}")
print(f"   • Model Portfolio Sharpe: {portfolio_sharpe:.4f} vs SPY: {spy_sharpe:.4f}")
print(f"   • Model Portfolio Max DD: {portfolio_max_dd:.2%} vs SPY: {spy_max_dd:.2%}")
print(f"   • Alpha: {portfolio_alpha:.2%}")
print(f"   • Beta: {portfolio_beta:.4f}")
print(f"   • $10,000 → ${portfolio_final_value:,.0f} (Model) vs ${spy_final_value:,.0f} (SPY)")
print("\n" + "="*80)